
> Developed a production-ready NLP API using FastAPI and OpenAI for product review summarization.  
> Implemented intelligent caching with hash-based invalidation and cost optimization logic.  
> Containerized the service using Docker and Docker Compose for easy deployment.


<div dir="rtl">

### مرحله ۱: راه‌اندازی نوت‌بوک

#### در این مرحله نوت‌بوک را آماده می‌کنیم، کتابخانه‌های اصلی را وارد می‌کنیم و ساختار اولیه برای خواندن داده را ایجاد می‌کنیم.

</div>

In [1]:
#!pip install openai
#!pip install python-dotenv


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)

print("Notebook is ready.")

file_path = "../data/raw/comments.csv"

try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully.")
    print("Data shape:", df.shape)
    display(df.head())
except FileNotFoundError:
    print("comments.csv not found yet.")

Notebook is ready.
Data loaded successfully.
Data shape: (25, 3)


,comment_id,product_id,comment_text
0,1,74123979,عالی به موقع با کیفیت
1,2,74123979,بسته بندی خوب ،ارسال به موقع ،ومشتری مداری
2,3,74123979,بسته بندی نسبت به خرید قبلی بی کیفیت شده بود قبلا جعبه شیک تری همراه سکه بود
3,4,74123979,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی ام
4,5,74123979,خوب


<div dir="rtl">

### مرحله ۲: بررسی اولیه داده‌ها

#### در این مرحله ساختار داده‌ها را بررسی می‌کنیم، ستون‌ها را می‌بینیم، داده‌های ناقص را شناسایی می‌کنیم و چند نمونه کامنت را تحلیل اولیه می‌کنیم.

</div>

In [3]:
if "df" in locals():

    print("Shape of data:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nSample comments:")
    display(df["comment_text"].sample(5))

Shape of data:
(25, 3)

Columns:
Index(['comment_id', 'product_id', 'comment_text'], dtype='object')

Missing values:
comment_id      0
product_id      0
comment_text    0
dtype: int64

Sample comments:


20                                                               ممنون مرسی متشکرم
2     بسته بندی نسبت به خرید قبلی بی کیفیت شده بود قبلا جعبه شیک تری همراه سکه بود
11                                                                  از خریدم راضیم
24                                                  تحویل با بسته بندی زیبا و سریع
10                                              تحویل به موقع و در بسته بندی مناسب
Name: comment_text, dtype: object

In [4]:
df = pd.read_csv("../data/raw/comments.csv")
display(df.head())

,comment_id,product_id,comment_text
0,1,74123979,عالی به موقع با کیفیت
1,2,74123979,بسته بندی خوب ،ارسال به موقع ،ومشتری مداری
2,3,74123979,بسته بندی نسبت به خرید قبلی بی کیفیت شده بود قبلا جعبه شیک تری همراه سکه بود
3,4,74123979,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی ام
4,5,74123979,خوب


<div dir="rtl">

### مرحله ۳: پاک‌سازی متن فارسی

#### در این مرحله متن کامنت‌ها را تمیز می‌کنیم، نویزها را حذف می‌کنیم و آن‌ها را برای تحلیل‌های بعدی آماده می‌کنیم.

</div>

In [5]:
import re

def clean_text(text):
    text = str(text)

    # حذف علائم نگارشی فارسی و انگلیسی
    text = re.sub(r"[،.!؟:؛,]", "", text)

    # حذف کاراکترهای غیر فارسی
    text = re.sub(r"[^\u0600-\u06FF\s]", "", text)

    # حذف فاصله‌های اضافی
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_comment"] = df["comment_text"].apply(clean_text)

<div dir="rtl">

### مرحله ۴: نرمال‌سازی متن فارسی

#### در این مرحله متن‌ها را استاندارد می‌کنیم تا حالت‌های مختلف نوشتاری (مثل ي/ی، ك/ک، فاصله‌ها و ...) یکسان شوند.

</div>

In [6]:
# pip install hazm

In [7]:
from hazm import Normalizer

normalizer = Normalizer()

df["normalized_comment"] = df["clean_comment"].apply(normalizer.normalize)

display(df[["clean_comment", "normalized_comment"]].head(10))

,clean_comment,normalized_comment
0,عالی به موقع با کیفیت,عالی به موقع با کیفیت
1,بسته بندی خوب ارسال به موقع ومشتری مداری,بسته بندی خوب ارسال به موقع ومشتری مداری
2,بسته بندی نسبت به خرید قبلی بی کیفیت شده بود قبلا جعبه شیک تری همراه سکه بود,بسته بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود
3,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی ام,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی‌ام
4,خوب,خوب
5,عالی بسته بندیشم خیلی قشنگ بود,عالی بسته بندیشم خیلی قشنگ بود
6,عالی بود,عالی بود
7,کرایه کمتر و سرعت ارسال,کرایه کمتر و سرعت ارسال
8,ممنون از ارسال به موقع و بسته بندی زیباتون,ممنون از ارسال به موقع و بسته بندی زیباتون
9,ارسال طبق عکس,ارسال طبق عکس


<div dir="rtl">

### مرحله ۵: تحلیل اولیه روی کامنت‌ها

#### در این مرحله طول کامنت‌ها، تعداد آن‌ها و چند ویژگی پایه‌ای را بررسی می‌کنیم تا شناخت بهتری از داده‌ها پیدا کنیم.

</div>

In [8]:
df["comment_length"] = df["normalized_comment"].apply(len)
df["word_count"] = df["normalized_comment"].apply(lambda x: len(x.split()))

print("تعداد کل کامنت‌ها:", len(df))
print("میانگین طول کامنت:", round(df["comment_length"].mean(), 2))
print("میانگین تعداد کلمات:", round(df["word_count"].mean(), 2))
print("کوتاه‌ترین کامنت:", df["comment_length"].min())
print("بلندترین کامنت:", df["comment_length"].max())

display(
    df[["normalized_comment", "comment_length", "word_count"]]
    .sort_values(by="comment_length", ascending=False)
    .head(10)
)

تعداد کل کامنت‌ها: 25
میانگین طول کامنت: 30.48
میانگین تعداد کلمات: 6.56
کوتاه‌ترین کامنت: 3
بلندترین کامنت: 76


,normalized_comment,comment_length,word_count
2,بسته بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود,76,15
3,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی‌ام,74,14
21,کیفیت بسته بندی و قیمت و سرعت ارسال نسبت به رقبا بهتر بوده,58,13
18,زود و با بسته بندی خوب همراه با فاکتور بدستم رسید,49,11
8,ممنون از ارسال به موقع و بسته بندی زیباتون,42,9
22,بسته بندیش عالی بود واقعا باکس ارسال میشه,41,8
12,بسیار عالی با بسته بندی شیک و ارسال سریع,40,9
1,بسته بندی خوب ارسال به موقع ومشتری مداری,40,8
10,تحویل به موقع و در بسته بندی مناسب,34,8
19,بسته بندی خوب و زود بدستم رسید,30,7


<div dir="rtl">

### مرحله ۶: استخراج پرتکرارترین کلمات

#### در این مرحله بررسی می‌کنیم که کاربران بیشتر از چه کلماتی استفاده کرده‌اند تا دید اولیه‌ای از موضوعات پرتکرار بگیریم.

</div>

In [9]:
from collections import Counter

all_words = " ".join(df["normalized_comment"]).split()

word_counts = Counter(all_words)

common_words = word_counts.most_common(20)

for word, count in common_words:
    print(f"{word} : {count}")

بسته : 12
و : 11
بندی : 10
به : 8
عالی : 7
ارسال : 7
بود : 7
با : 6
خوب : 5
موقع : 4
نسبت : 3
خرید : 3
بسیار : 3
سریع : 3
رسید : 3
کیفیت : 2
همراه : 2
خیلی : 2
سرعت : 2
ممنون : 2


<div dir="rtl">

### مرحله ۷: حذف کلمات بی‌اهمیت (Stopwords)

#### در این مرحله کلمات پرتکرار ولی بی‌معنی مثل «و»، «از»، «به» را حذف می‌کنیم تا تحلیل دقیق‌تری از متن داشته باشیم.

</div>

In [10]:
from hazm import stopwords_list
from collections import Counter

stopwords = set(stopwords_list())

# حذف کلمات مهم از stopwords
important_words = ["خوب", "عالی", "بد", "گرونه"]

for word in important_words:
    if word in stopwords:
        stopwords.remove(word)

filtered_words = [
    word for word in all_words
    if word not in stopwords and len(word) > 1
]

word_counts = Counter(filtered_words)

common_words = word_counts.most_common(20)

for word, count in common_words:
    print(f"{word} : {count}")

بسته : 12
عالی : 7
ارسال : 7
خوب : 5
موقع : 4
خرید : 3
سریع : 3
کیفیت : 2
همراه : 2
سرعت : 2
ممنون : 2
تحویل : 2
شیک : 2
زود : 2
بدستم : 2
ومشتری : 1
مداری : 1
قبلی : 1
بی‌کیفیت : 1
قبلا : 1


<div dir="rtl">

### مرحله ۸: بهبود دیکشنری عبارت‌های مهم

#### در این مرحله فقط عبارت‌های دوکلمه‌ای مفید و معنادار را برای نرمال‌سازی انتخاب می‌کنیم تا متن‌ها استانداردتر شوند.

</div>

In [11]:
phrase_replacements = {
    "بسته بندی": "بسته‌بندی",
    "به موقع": "به‌موقع",
    "بدستم رسید": "به‌دستم رسید",
    "مشتری مداری": "مشتری‌مداری",
    "با کیفیت": "باکیفیت",
    "سرعت ارسال": "سرعت‌ارسال",
    "بسیار عالی": "بسیار‌عالی"
}

def normalize_phrases(text):
    text = str(text)
    for old, new in phrase_replacements.items():
        text = text.replace(old, new)
    return text

df["final_comment"] = df["normalized_comment"].apply(normalize_phrases)

display(df[["normalized_comment", "final_comment"]].head(10))

,normalized_comment,final_comment
0,عالی به موقع با کیفیت,عالی به‌موقع باکیفیت
1,بسته بندی خوب ارسال به موقع ومشتری مداری,بسته‌بندی خوب ارسال به‌موقع ومشتری‌مداری
2,بسته بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود,بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود
3,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی‌ام,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی‌ام
4,خوب,خوب
5,عالی بسته بندیشم خیلی قشنگ بود,عالی بسته‌بندیشم خیلی قشنگ بود
6,عالی بود,عالی بود
7,کرایه کمتر و سرعت ارسال,کرایه کمتر و سرعت‌ارسال
8,ممنون از ارسال به موقع و بسته بندی زیباتون,ممنون از ارسال به‌موقع و بسته‌بندی زیباتون
9,ارسال طبق عکس,ارسال طبق عکس


<div dir="rtl">

### مرحله ۸.۱: شناسایی خودکار ترکیب‌های پرتکرار

#### در این مرحله ترکیب‌های دوکلمه‌ای پرتکرار را استخراج می‌کنیم تا بتوانیم آن‌ها را به‌صورت هوشمند به سیستم اضافه کنیم.

</div>

In [12]:
from collections import Counter

# استخراج bigram (دو کلمه‌ای)
bigrams = []

for sentence in df["normalized_comment"]:
    words = sentence.split()
    for i in range(len(words) - 1):
        bigram = words[i] + " " + words[i+1]
        bigrams.append(bigram)

bigram_counts = Counter(bigrams)

common_bigrams = bigram_counts.most_common(20)

for phrase, count in common_bigrams:
    print(f"{phrase} : {count}")

بسته بندی : 10
به موقع : 4
با بسته : 4
بندی خوب : 3
نسبت به : 3
عالی بود : 3
ارسال به : 2
و سرعت : 2
سرعت ارسال : 2
موقع و : 2
بسیار عالی : 2
عالی با : 2
بندی شیک : 2
بدستم رسید : 2
عالی به : 1
موقع با : 1
با کیفیت : 1
خوب ارسال : 1
موقع ومشتری : 1
ومشتری مداری : 1


<div dir="rtl">

### مرحله ۸.۲: استخراج bigram از کلمات فیلترشده

#### در این مرحله bigramها را نه از متن خام، بلکه از کلمات فیلترشده می‌سازیم تا ترکیب‌های بی‌معنی مثل «و سرعت» یا «با بسته» حذف شوند.

</div>

In [13]:
from collections import Counter

bigrams_filtered = []

for sentence in df["final_comment"]:
    words = sentence.split()
    
    filtered_sentence_words = [
        word for word in words
        if word not in stopwords and len(word) > 1
    ]
    
    for i in range(len(filtered_sentence_words) - 1):
        bigram = filtered_sentence_words[i] + " " + filtered_sentence_words[i + 1]
        bigrams_filtered.append(bigram)

bigram_counts_filtered = Counter(bigrams_filtered)

for phrase, count in bigram_counts_filtered.most_common(20):
    print(f"{phrase} : {count}")

بسته‌بندی خوب : 3
ارسال به‌موقع : 2
به‌موقع بسته‌بندی : 2
بسیار‌عالی بسته‌بندی : 2
بسته‌بندی شیک : 2
عالی به‌موقع : 1
به‌موقع باکیفیت : 1
خوب ارسال : 1
به‌موقع ومشتری‌مداری : 1
بسته‌بندی خرید : 1
خرید قبلی : 1
قبلی بی‌کیفیت : 1
بی‌کیفیت قبلا : 1
قبلا جعبه : 1
جعبه شیک‌تری : 1
شیک‌تری همراه : 1
همراه سکه : 1
ازاین خرید : 1
خرید همینکه : 1
همینکه بصورت : 1


<div dir="rtl">

### مرحله ۸.۳: اصلاح واژه‌ها و عبارت‌های خام

#### در این مرحله چند واژه و عبارت پرتکرار ولی غیراستاندارد را اصلاح می‌کنیم تا خروجی تحلیل طبیعی‌تر و دقیق‌تر شود.

</div>

In [14]:
extra_replacements = {
    "ومشتری": "و مشتری",
    "ازاین": "از این",
    "همینکه": "همین که",
    "بصورت": "به‌صورت",
    "میتونم": "می‌توانم",
    "راضی‌ام": "راضی‌ام",
    "بندیشم": "بسته‌بندیش",
    "بدستم": "به‌دستم"
}

def refine_text(text):
    text = str(text)
    for old, new in extra_replacements.items():
        text = text.replace(old, new)
    return text

df["refined_comment"] = df["final_comment"].apply(refine_text)

display(df[["final_comment", "refined_comment"]].head(10))

,final_comment,refined_comment
0,عالی به‌موقع باکیفیت,عالی به‌موقع باکیفیت
1,بسته‌بندی خوب ارسال به‌موقع ومشتری‌مداری,بسته‌بندی خوب ارسال به‌موقع و مشتری‌مداری
2,بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود,بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود
3,من ازاین چند بسیار خرید کردم و همینکه بصورت اقساطی میتونم خرید کنم راضی‌ام,من از این چند بسیار خرید کردم و همین که به‌صورت اقساطی می‌توانم خرید کنم راضی‌ام
4,خوب,خوب
5,عالی بسته‌بندیشم خیلی قشنگ بود,عالی بسته‌بسته‌بندیش خیلی قشنگ بود
6,عالی بود,عالی بود
7,کرایه کمتر و سرعت‌ارسال,کرایه کمتر و سرعت‌ارسال
8,ممنون از ارسال به‌موقع و بسته‌بندی زیباتون,ممنون از ارسال به‌موقع و بسته‌بندی زیباتون
9,ارسال طبق عکس,ارسال طبق عکس


In [15]:
bigrams_refined = []

for sentence in df["refined_comment"]:
    words = sentence.split()

    filtered_sentence_words = [
        word for word in words
        if word not in stopwords and len(word) > 1
    ]

    for i in range(len(filtered_sentence_words) - 1):
        bigram = filtered_sentence_words[i] + " " + filtered_sentence_words[i + 1]
        bigrams_refined.append(bigram)

bigram_counts_refined = Counter(bigrams_refined)

for phrase, count in bigram_counts_refined.most_common(20):
    print(f"{phrase} : {count}")

بسته‌بندی خوب : 3
ارسال به‌موقع : 2
به‌موقع بسته‌بندی : 2
بسیار‌عالی بسته‌بندی : 2
بسته‌بندی شیک : 2
عالی به‌موقع : 1
به‌موقع باکیفیت : 1
خوب ارسال : 1
به‌موقع مشتری‌مداری : 1
بسته‌بندی خرید : 1
خرید قبلی : 1
قبلی بی‌کیفیت : 1
بی‌کیفیت قبلا : 1
قبلا جعبه : 1
جعبه شیک‌تری : 1
شیک‌تری همراه : 1
همراه سکه : 1
خرید به‌صورت : 1
به‌صورت اقساطی : 1
اقساطی می‌توانم : 1


<div dir="rtl">

### مرحله ۹: استخراج insight برای فروشنده

#### در این مرحله سعی می‌کنیم به‌صورت ساده، نکات مثبت و منفی را از کامنت‌ها استخراج کنیم.

</div>

In [16]:
positive_keywords = ["عالی", "خوب", "سریع", "شیک", "راضی"]
negative_keywords = ["بد", "بی‌کیفیت", "گرون", "مشکل"]

positive_comments = []
negative_comments = []

for comment in df["refined_comment"]:
    
    if any(word in comment for word in positive_keywords):
        positive_comments.append(comment)
        
    if any(word in comment for word in negative_keywords):
        negative_comments.append(comment)

print("تعداد کامنت‌های مثبت:", len(positive_comments))
print("تعداد کامنت‌های منفی:", len(negative_comments))

print("\nنمونه کامنت‌های مثبت:")
for c in positive_comments[:5]:
    print("-", c)

print("\nنمونه کامنت‌های منفی:")
for c in negative_comments[:5]:
    print("-", c)

تعداد کامنت‌های مثبت: 17
تعداد کامنت‌های منفی: 3

نمونه کامنت‌های مثبت:
- عالی به‌موقع باکیفیت
- بسته‌بندی خوب ارسال به‌موقع و مشتری‌مداری
- بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود
- من از این چند بسیار خرید کردم و همین که به‌صورت اقساطی می‌توانم خرید کنم راضی‌ام
- خوب

نمونه کامنت‌های منفی:
- بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود
- سلام خوب بود بد نبود
- نسبت به بازار گرونه


<div dir="rtl">

### مرحله ۹.۱: بهبود تشخیص احساس

#### در این مرحله به جای تشخیص ساده، تعداد کلمات مثبت و منفی را در هر کامنت می‌شماریم و تصمیم دقیق‌تری می‌گیریم.

</div>

In [17]:
positive_keywords = ["عالی", "خوب", "سریع", "شیک", "راضی"]
negative_keywords = ["بد", "بی‌کیفیت", "گرون", "مشکل"]

def get_sentiment(comment):
    pos_count = sum(word in comment for word in positive_keywords)
    neg_count = sum(word in comment for word in negative_keywords)
    
    if pos_count > neg_count:
        return "positive"
    elif neg_count > pos_count:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["refined_comment"].apply(get_sentiment)

print(df["sentiment"].value_counts())

display(df[["refined_comment", "sentiment"]].head(10))

sentiment
positive    15
neutral      9
negative     1
Name: count, dtype: int64


,refined_comment,sentiment
0,عالی به‌موقع باکیفیت,positive
1,بسته‌بندی خوب ارسال به‌موقع و مشتری‌مداری,positive
2,بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود,neutral
3,من از این چند بسیار خرید کردم و همین که به‌صورت اقساطی می‌توانم خرید کنم راضی‌ام,positive
4,خوب,positive
5,عالی بسته‌بسته‌بندیش خیلی قشنگ بود,positive
6,عالی بود,positive
7,کرایه کمتر و سرعت‌ارسال,neutral
8,ممنون از ارسال به‌موقع و بسته‌بندی زیباتون,neutral
9,ارسال طبق عکس,neutral


In [18]:
df["refined_comment"] = df["refined_comment"].str.replace("بسته‌بسته‌بندیش", "بسته‌بندیش")

<div dir="rtl">

### مرحله ۱۰: خلاصه‌سازی نظرات کاربران

#### در این مرحله تلاش می‌کنیم از مجموعه کامنت‌ها یک خلاصه کلی و قابل فهم استخراج کنیم.

</div>

In [19]:
top_phrases = [phrase for phrase, count in bigram_counts_refined.most_common(5)]

print("خلاصه نظرات کاربران:\n")

for phrase in top_phrases:
    print("-", phrase)

خلاصه نظرات کاربران:

- بسته‌بندی خوب
- ارسال به‌موقع
- به‌موقع بسته‌بندی
- بسیار‌عالی بسته‌بندی
- بسته‌بندی شیک


<div dir="rtl">

### مرحله ۱۱.۱: آماده‌سازی داده برای LLM

#### در این مرحله کامنت‌ها را به یک متن واحد تبدیل می‌کنیم تا به مدل زبانی بدهیم.

</div>

In [20]:
comments_text = "\n".join(df["refined_comment"].tolist())

print(comments_text[:1000])


عالی به‌موقع باکیفیت
بسته‌بندی خوب ارسال به‌موقع و مشتری‌مداری
بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود
من از این چند بسیار خرید کردم و همین که به‌صورت اقساطی می‌توانم خرید کنم راضی‌ام
خوب
عالی بسته‌بندیش خیلی قشنگ بود
عالی بود
کرایه کمتر و سرعت‌ارسال
ممنون از ارسال به‌موقع و بسته‌بندی زیباتون
ارسال طبق عکس
تحویل به‌موقع و در بسته‌بندی مناسب
از خریدم راضیم
بسیار‌عالی با بسته‌بندی شیک و ارسال سریع
خیلی سریع به دستم رسید و خوبه
همیشه عالی بود
سلام خوب بود بد نبود
بسیار‌عالی با بسته‌بندی شیک
موفق باشید
زود و با بسته‌بندی خوب همراه با فاکتور به‌دستم رسید
بسته‌بندی خوب و زود به‌دستم رسید
ممنون مرسی متشکرم
کیفیت بسته‌بندی و قیمت و سرعت‌ارسال نسبت به رقبا بهتر بوده
بسته‌بندیش عالی بود واقعا باکس ارسال میشه
نسبت به بازار گرونه
تحویل با بسته‌بندی زیبا و سریع


<div dir="rtl">

### مرحله ۱۱.۲: طراحی Prompt برای خلاصه‌سازی

#### در این مرحله یک دستور دقیق به مدل می‌دهیم تا خروجی حرفه‌ای تولید کند.

</div>

In [21]:
prompt = f"""
شما یک تحلیل‌گر حرفه‌ای تجربه مشتری هستید.

نظرات کاربران درباره یک محصول را در ادامه می‌بینید:

{comments_text}

لطفاً:
1. مهم‌ترین نقاط قوت را استخراج کن
2. مهم‌ترین نقاط ضعف را استخراج کن
3. یک جمع‌بندی کلی بنویس

پاسخ را به صورت خلاصه، شفاف و حرفه‌ای به زبان فارسی بنویس.
"""

<div dir="rtl">

### مرحله ۱۱.۴: اتصال امن به API

#### در این مرحله کلید API را از فایل env می‌خوانیم و اتصال به مدل زبانی را برقرار می‌کنیم.

</div>

In [24]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("API Connected Successfully ✅")

API Connected Successfully ✅


<div dir="rtl">

### مرحله ۱۱.۵: اجرای خلاصه‌سازی با LLM

#### در این مرحله متن کامنت‌ها را به مدل زبانی می‌دهیم تا نقاط قوت، نقاط ضعف و جمع‌بندی کلی را به‌صورت حرفه‌ای تولید کند.

</div>

In [23]:
comments_text = "\n".join(df["refined_comment"].tolist())

prompt = f"""
شما یک تحلیل‌گر حرفه‌ای تجربه مشتری هستید.

در ادامه، نظرات کاربران درباره یک محصول را می‌بینید:

{comments_text}

لطفاً فقط بر اساس همین نظرات:

1. مهم‌ترین نقاط قوت را استخراج کن
2. مهم‌ترین نقاط ضعف را استخراج کن
3. یک جمع‌بندی کلی 3 تا 5 خطی بنویس

قالب پاسخ دقیقاً این‌طور باشد:

نقاط قوت:
- ...
- ...

نقاط ضعف:
- ...
- ...

جمع‌بندی:
...

پاسخ را حرفه‌ای، کوتاه، شفاف و به زبان فارسی بنویس.
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

llm_summary = response.choices[0].message.content
print(llm_summary)

نقاط قوت:  
- ارسال به‌موقع و سریع  
- بسته‌بندی زیبا، شیک و باکیفیت  
- مشتری‌مداری و امکان خرید اقساطی  
- کیفیت محصول قابل قبول و مطابق با عکس  

نقاط ضعف:  
- افزایش قیمت نسبت به بازار  
- کاهش کیفیت بسته‌بندی نسبت به خریدهای قبلی (جعبه و همراهی سکه)  

جمع‌بندی:  
نظرات کاربران نشان‌دهنده رضایت کلی از کیفیت محصول، بسته‌بندی شیک و ارسال سریع است که از مهم‌ترین نقاط قوت محسوب می‌شود. با این حال، برخی کاربران به افزایش قیمت و افت کیفیت بسته‌بندی نسبت به قبل اشاره کرده‌اند که می‌تواند بهبود یابد. به طور کلی تجربه مشتری مثبت و قابل قبول ارزیابی می‌شود.


<div dir="rtl">

### مرحله ۱۳: اجرای مدل واقعی sentiment

#### در این مرحله به‌جای روش rule-based، از یک مدل ازپیش‌آموزش‌دیده فارسی برای تشخیص احساس کامنت‌ها استفاده می‌کنیم.

</div>

In [25]:
!pip install transformers torch accelerate

  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 4.5 MB/s  0:00:02 4.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 1.5 MB/s  0:00:005.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 4.5 MB/s  0:00:004.4 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 6.5 MB/s  0:00:006.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 3.0 MB/s  0:00:26 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.4 MB/s  0:00:00m 4.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 3.5 MB/s  0:00:01m 3.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 1.4 MB/s  0:00:00m ? eta -:--:--
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB

<div dir="rtl">

### مرحله ۱۳.۱: لود کردن مدل sentiment فارسی

#### در این مرحله مدل فارسی مناسب برای تحلیل احساس کامنت‌های محصول را لود می‌کنیم.

</div>

In [26]:
from transformers import pipeline

sentiment_model_name = "HooshvareLab/bert-fa-base-uncased-sentiment-digikala"

sentiment_pipe = pipeline(
    "text-classification",
    model=sentiment_model_name,
    tokenizer=sentiment_model_name
)

print("Sentiment model loaded successfully ✅")

/Users/rasoolzandian/inchand_nlp_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████| 201/201 [00:00<00:00, 38119.69it/s]
BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased-sentiment-digikala
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded successfully ✅


<div dir="rtl">

### مرحله ۱۳.۲: گرفتن پیش‌بینی برای همه کامنت‌ها

#### در این مرحله مدل را روی کامنت‌های واقعی خودمان اجرا می‌کنیم و برچسب احساس هر کامنت را می‌گیریم.

</div>

In [27]:
texts = df["refined_comment"].fillna("").tolist()

predictions = sentiment_pipe(texts)

df["hf_label"] = [item["label"] for item in predictions]
df["hf_score"] = [item["score"] for item in predictions]

display(df[["refined_comment", "hf_label", "hf_score"]].head(10))

,refined_comment,hf_label,hf_score
0,عالی به‌موقع باکیفیت,recommended,0.953117
1,بسته‌بندی خوب ارسال به‌موقع و مشتری‌مداری,recommended,0.955662
2,بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود,not_recommended,0.891067
3,من از این چند بسیار خرید کردم و همین که به‌صورت اقساطی می‌توانم خرید کنم راضی‌ام,recommended,0.957755
4,خوب,recommended,0.841696
5,عالی بسته‌بندیش خیلی قشنگ بود,recommended,0.970884
6,عالی بود,recommended,0.932885
7,کرایه کمتر و سرعت‌ارسال,no_idea,0.420960
8,ممنون از ارسال به‌موقع و بسته‌بندی زیباتون,recommended,0.954429
9,ارسال طبق عکس,not_recommended,0.480765


<div dir="rtl">

### مرحله ۱۳.۳: خواناتر کردن برچسب‌ها

#### در این مرحله برچسب‌های مدل را به معادل‌های قابل فهم فارسی تبدیل می‌کنیم.

</div>

In [29]:
label_map = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive",
    "not_recommended": "negative",
    "no_idea": "neutral",
    "recommended": "positive"
}

df["sentiment_real"] = df["hf_label"].map(lambda x: label_map.get(x, x))

display(df[["refined_comment", "hf_label", "sentiment_real", "hf_score"]].head(25))
print(df["sentiment_real"].value_counts())

,refined_comment,hf_label,sentiment_real,hf_score
0,عالی به‌موقع باکیفیت,recommended,positive,0.953117
1,بسته‌بندی خوب ارسال به‌موقع و مشتری‌مداری,recommended,positive,0.955662
2,بسته‌بندی نسبت به خرید قبلی بی‌کیفیت شده بود قبلا جعبه شیک‌تری همراه سکه بود,not_recommended,negative,0.891067
3,من از این چند بسیار خرید کردم و همین که به‌صورت اقساطی می‌توانم خرید کنم راضی‌ام,recommended,positive,0.957755
4,خوب,recommended,positive,0.841696
5,عالی بسته‌بندیش خیلی قشنگ بود,recommended,positive,0.970884
6,عالی بود,recommended,positive,0.932885
7,کرایه کمتر و سرعت‌ارسال,no_idea,neutral,0.420960
8,ممنون از ارسال به‌موقع و بسته‌بندی زیباتون,recommended,positive,0.954429
9,ارسال طبق عکس,not_recommended,negative,0.480765


sentiment_real
positive    19
negative     3
neutral      3
Name: count, dtype: int64


<div dir="rtl">

### مرحله ۱۳.۴: ذخیره نتیجه مدل

#### در این مرحله خروجی sentiment واقعی را برای استفاده بعدی ذخیره می‌کنیم.

</div>

In [30]:
output_path = "../data/processed/comments_with_real_sentiment.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Real sentiment output saved successfully ✅")

Real sentiment output saved successfully ✅


<div dir="rtl">

### مرحله ۱۵: طراحی دقیق Contract API

#### در این مرحله ورودی‌ها، خروجی‌ها، status codeها و رفتار دقیق endpointهای API را مشخص می‌کنیم تا تیم فنی بداند دقیقاً چگونه باید با سرویس کار کند.

</div>

<div dir="rtl">

### خلاصه معماری کلی API

#### هدف
ساخت یک API هوشمند برای خلاصه‌سازی نظرات محصولات که فقط در صورت تغییر کامنت‌ها، دوباره از LLM استفاده کند و در غیر این صورت خلاصه ذخیره‌شده قبلی را برگرداند.

---

### منطق کلی سیستم

تیم فنی با ارسال `product_id` به API، خلاصه محصول را درخواست می‌کند.  
API ابتدا بررسی می‌کند که آیا برای این محصول خلاصه معتبر و به‌روز وجود دارد یا نه.

- اگر خلاصه معتبر باشد → همان نسخه ذخیره‌شده برگردانده می‌شود
- اگر خلاصه وجود نداشته باشد یا داده تغییر کرده باشد → خلاصه جدید ساخته می‌شود، ذخیره می‌شود و سپس برگردانده می‌شود

---

### اجزای اصلی سیستم

#### 1. لایه داده کامنت‌ها
مسئول خواندن کامنت‌های هر محصول از منبع داده است.  
در نسخه MVP می‌تواند فایل CSV باشد و در نسخه واقعی می‌تواند دیتابیس یا API داخلی تیم فنی باشد.

#### 2. لایه تشخیص تغییر
بررسی می‌کند که آیا کامنت‌های محصول نسبت به آخرین خلاصه تغییر کرده‌اند یا نه.  
این بررسی با استفاده از:
- تعداد کامنت‌ها
- تاریخ آخرین کامنت
- و مهم‌تر از همه `comments_hash`
انجام می‌شود.

#### 3. لایه خلاصه‌سازی
اگر داده جدید باشد، این بخش با استفاده از LLM:
- نقاط قوت
- نقاط ضعف
- جمع‌بندی کلی
را تولید می‌کند.

#### 4. لایه ذخیره‌سازی خلاصه
خلاصه آخر هر محصول و اطلاعات مربوط به آن را ذخیره می‌کند تا در درخواست‌های بعدی بدون هزینه اضافی قابل استفاده باشد.

---

### منطق Cache

برای هر `product_id` یک رکورد summary نگهداری می‌شود.  
اگر `comments_hash` جدید با مقدار ذخیره‌شده برابر باشد، یعنی کامنت‌ها تغییری نکرده‌اند و نیازی به ساخت خلاصه جدید نیست.

اگر hash تغییر کرده باشد، یعنی:
- کامنت جدید آمده
- یا کامنتی ویرایش شده
- یا کامنتی حذف شده

در این صورت summary جدید ساخته می‌شود.

---

### اطلاعات ذخیره‌شده برای هر محصول

برای هر محصول، این اطلاعات نگهداری می‌شود:

- `product_id`
- `summary`
- `pros`
- `cons`
- `sentiment_overview`
- `comment_count`
- `last_comment_date`
- `comments_hash`
- `last_generated_at`
- `status`

---

### Endpointهای اصلی API

- `GET /api/v1/health`  
  برای بررسی فعال بودن سرویس

- `GET /api/v1/summary/{product_id}`  
  برای گرفتن summary محصول

- `POST /api/v1/refresh/{product_id}`  
  برای بازسازی اجباری summary

- `GET /api/v1/summary/{product_id}/status`  
  برای بررسی وضعیت summary بدون دریافت متن کامل

---

### رفتار کلی Endpoint اصلی

#### اگر summary معتبر باشد:
- از cache خوانده می‌شود
- سریع برگردانده می‌شود
- هزینه LLM ندارد

#### اگر summary وجود نداشته باشد یا stale باشد:
- کامنت‌ها دوباره خوانده می‌شوند
- summary جدید ساخته می‌شود
- ذخیره می‌شود
- به کاربر برگردانده می‌شود

---

### مزایای این معماری

- کاهش هزینه استفاده از LLM
- افزایش سرعت پاسخ
- جلوگیری از callهای تکراری و غیرضروری
- آماده برای استفاده تیم فنی در محصول واقعی
- قابل توسعه به نسخه production-level

---

### جمع‌بندی

این API فقط یک wrapper ساده روی LLM نیست، بلکه یک **Summary Service هوشمند** است که:
- داده را می‌خواند
- تغییر را تشخیص می‌دهد
- فقط در صورت نیاز خلاصه جدید می‌سازد
- و در بقیه مواقع از cache استفاده می‌کند

</div>

<div dir="rtl">

### مرحله ۱۶: طراحی ساختار فایل‌های پروژه برای API

#### در این مرحله ساختار پوشه‌ها و فایل‌های لازم برای پیاده‌سازی API را مشخص می‌کنیم تا پروژه از حالت آزمایشی به سمت یک سرویس واقعی و قابل توسعه حرکت کند.

</div>

<div dir="rtl">

### مرحله ۱۷: راه‌اندازی اولیه FastAPI

#### هدف این مرحله
در این مرحله اولین نسخه از API را با استفاده از فریم‌ورک FastAPI راه‌اندازی می‌کنیم تا زیرساخت لازم برای ارائه سرویس تحلیل کامنت‌ها فراهم شود.

---

#### آنچه در این مرحله انجام شد

- نصب فریم‌ورک FastAPI و سرور ASGI (uvicorn)
- ایجاد فایل اصلی API در مسیر `src/api/main.py`
- ساخت شیء اصلی اپلیکیشن با استفاده از `FastAPI()`
- تعریف اولین endpoint به نام `/api/v1/health`
- اجرای سرور به‌صورت محلی
- تست endpoint از طریق مرورگر و ابزار Swagger

---

#### توضیح فنی

FastAPI یک فریم‌ورک مدرن برای ساخت API در پایتون است که بر پایه استاندارد ASGI کار می‌کند و از type hintهای پایتون برای اعتبارسنجی داده‌ها استفاده می‌کند.

در این مرحله:

- با استفاده از `FastAPI()` یک اپلیکیشن API ساخته شد
- endpoint `/health` برای بررسی وضعیت سرویس (health check) تعریف شد
- با استفاده از `uvicorn` سرور به‌صورت محلی اجرا شد
- از قابلیت Swagger UI (مسیر `/docs`) برای تست و مشاهده API استفاده شد

---

#### اهمیت این مرحله

این مرحله پایه کل سیستم است، زیرا:

- امکان تست و توسعه endpointهای بعدی را فراهم می‌کند
- ارتباط بین کلاینت (تیم فنی) و سرویس را برقرار می‌کند
- زیرساخت لازم برای اضافه کردن منطق NLP و LLM را آماده می‌سازد

---

#### نتیجه

در پایان این مرحله، یک API فعال داریم که:

- روی لوکال اجرا می‌شود
- درخواست HTTP دریافت می‌کند
- پاسخ JSON برمی‌گرداند
- و آماده توسعه endpointهای واقعی مانند `/summary` است

</div>

<div dir="rtl">

### مرحله ۱۸: اضافه کردن endpoint واقعی summary

#### در این مرحله endpoint اصلی API را می‌سازیم تا با گرفتن شناسه محصول، کامنت‌های آن محصول را از فایل بخواند و یک پاسخ ساختاریافته برگرداند.

</div>

<div dir="rtl">

### مرحله ۱۹: انتقال منطق summary به summary_service

#### در این مرحله منطق ساخت خلاصه از فایل `main.py` جدا می‌شود و به یک سرویس مستقل منتقل می‌شود تا ساختار پروژه تمیزتر، قابل توسعه‌تر و مناسب‌تر برای نسخه production شود.

---

#### آنچه در این مرحله انجام می‌شود

- ایجاد فایل `summary_service.py`
- انتقال منطق ساخت خلاصه اولیه به این فایل
- ساده‌تر کردن endpoint در `main.py`
- آماده‌سازی پروژه برای اتصال بعدی به LLM و cache

---

#### اهمیت این مرحله

در معماری تمیز، فایل API نباید شامل منطق بیزینسی باشد.  
API فقط باید درخواست را بگیرد و پاسخ را برگرداند.  
منطق اصلی باید در service layer قرار بگیرد تا:

- نگهداری کد آسان‌تر شود
- تست‌پذیری بیشتر شود
- اضافه کردن cache و LLM ساده‌تر شود

</div>

<div dir="rtl">

### مرحله ۲۰: ساخت file_store و ذخیره summaryها

#### در این مرحله یک لایه ذخیره‌سازی ساده ایجاد می‌کنیم تا خلاصه‌های ساخته‌شده برای هر محصول نگهداری شوند و در درخواست‌های بعدی، در صورت معتبر بودن، دوباره استفاده شوند.

---

#### هدف این مرحله

- جلوگیری از تولید تکراری summary
- کاهش هزینه و زمان پاسخ
- آماده‌سازی پروژه برای cache-aware behavior
- نزدیک شدن به نسخه production-level

---

#### اهمیت این مرحله

بدون ذخیره‌سازی، API در هر درخواست باید دوباره summary بسازد.  
اما با داشتن file store، می‌توانیم آخرین نسخه summary هر محصول را نگه داریم و فقط در صورت تغییر داده‌ها آن را بازسازی کنیم.

</div>

<div dir="rtl">

### مرحله ۲۱: استفاده از cache و برگرداندن summary ذخیره‌شده

#### در این مرحله API قبل از ساخت summary جدید، ابتدا storage را بررسی می‌کند. اگر برای محصول یک summary ذخیره‌شده و معتبر وجود داشته باشد، همان نسخه قبلی برگردانده می‌شود و از ساخت مجدد summary جلوگیری می‌شود.

---

#### هدف این مرحله

- جلوگیری از بازتولید غیرضروری summary
- افزایش سرعت پاسخ API
- کاهش هزینه و مصرف منابع
- پیاده‌سازی اولین نسخه واقعی cache behavior

---

#### اهمیت این مرحله

تا قبل از این، summary فقط ذخیره می‌شد.  
اما از این مرحله به بعد، API از summary ذخیره‌شده هم استفاده می‌کند.  
این یعنی اگر محصول قبلاً خلاصه شده باشد، درخواست بعدی می‌تواند بدون ساخت summary جدید پاسخ داده شود.

</div>

<div dir="rtl">

### مرحله ۲۲: ساخت hash_service و اعتبارسنجی cache

#### در این مرحله برای هر محصول یک امضای یکتا (hash) از مجموعه کامنت‌ها می‌سازیم تا API بتواند تشخیص دهد آیا داده‌های محصول نسبت به آخرین summary تغییر کرده‌اند یا نه.

---

#### هدف این مرحله

- تشخیص دقیق تغییر واقعی کامنت‌ها
- جلوگیری از استفاده از cache قدیمی
- بازسازی summary فقط در صورت نیاز
- نزدیک شدن به رفتار production-level

---

#### اهمیت این مرحله

اگر فقط بر اساس وجود یا عدم وجود summary تصمیم بگیریم، ممکن است summary قدیمی برگردانده شود، حتی اگر کامنت جدید اضافه شده یا متن کامنتی ویرایش شده باشد.  
با استفاده از hash، می‌توانیم مطمئن شویم که summary همیشه با نسخه فعلی کامنت‌ها سازگار است.

</div>

<div dir="rtl">

### مرحله ۲۳: اتصال LLM واقعی به API

#### در این مرحله منطق ارتباط با مدل زبانی را به یک سرویس مستقل منتقل می‌کنیم تا API بتواند به‌جای خلاصه‌سازی ساده، summary حرفه‌ای، نقاط قوت و نقاط ضعف را با استفاده از LLM تولید کند.

---

#### هدف این مرحله

- ساخت فایل `llm_service.py`
- تعریف prompt استاندارد برای خلاصه‌سازی
- دریافت خروجی ساختاریافته از LLM
- استفاده از خروجی LLM در `summary_service`

---

#### اهمیت این مرحله

این مرحله API را از یک نسخه آزمایشی به یک سرویس هوشمند واقعی نزدیک می‌کند، چون summary دیگر فقط کنار هم گذاشتن چند کامنت نیست، بلکه یک تحلیل حرفه‌ای و قابل ارائه خواهد بود.

</div>

<div dir="rtl">

### مرحله ۲۴: مدیریت خطاهای LLM و مقاوم‌سازی API

#### در این مرحله خطاهای احتمالی در ارتباط با مدل زبانی را مدیریت می‌کنیم تا API در شرایط واقعی پایدارتر، قابل اعتمادتر و مناسب استفاده تیم فنی باشد.

---

#### هدف این مرحله

- جلوگیری از crash شدن API در صورت خطای LLM
- مدیریت پاسخ‌های نامعتبر یا JSON خراب
- ایجاد fallback مناسب
- آماده‌سازی برای استفاده production-like

---

#### اهمیت این مرحله

در پروژه‌های واقعی، مدل زبانی همیشه پاسخ کامل و بی‌نقص نمی‌دهد.  
برای همین API باید بتواند در شرایط خطا هم رفتار قابل پیش‌بینی داشته باشد و پاسخ کنترل‌شده برگرداند.

</div>

<div dir="rtl">

### مرحله ۲۵: اضافه کردن endpointهای status و refresh

#### در این مرحله دو endpoint مهم به API اضافه می‌کنیم: یکی برای بررسی وضعیت summary هر محصول و دیگری برای بازسازی اجباری summary، بدون توجه به cache.

---

#### هدف این مرحله

- کامل‌تر کردن contract API
- دادن کنترل بیشتر به تیم فنی
- امکان مانیتورینگ وضعیت summary
- امکان refresh دستی در مواقع خاص

---

#### اهمیت این مرحله

در استفاده واقعی، تیم فنی همیشه فقط summary نهایی را نمی‌خواهد.  
گاهی لازم است بداند وضعیت یک محصول چیست، آیا summary آماده است یا نه، و گاهی هم لازم است summary را به‌صورت اجباری دوباره بسازد.  
این دو endpoint دقیقاً برای همین نیازها طراحی می‌شوند.

</div>

<div dir="rtl">

### مرحله ۲۶: تعیین حداقل تعداد کامنت برای خلاصه‌سازی

#### در این مرحله یک قانون منطقی به سیستم اضافه می‌کنیم که فقط در صورت کافی بودن تعداد کامنت‌ها، خلاصه‌سازی با LLM انجام شود.

---

#### هدف این مرحله

- جلوگیری از استفاده غیرضروری از LLM
- بهبود کیفیت خروجی API
- ارائه پاسخ منطقی برای محصولات با کامنت کم
- کنترل هزینه و منابع

---

#### منطق تصمیم‌گیری

بر اساس تعداد کامنت‌ها:

- 0 کامنت → no_comments
- 1 کامنت → single_comment
- 2 کامنت → few_comments
- 3+ کامنت → استفاده از LLM برای خلاصه‌سازی

---

#### اهمیت این مرحله

خلاصه‌سازی زمانی ارزش دارد که داده کافی وجود داشته باشد.  
برای تعداد کم کامنت، نمایش مستقیم نظرات یا preview ساده، هم دقیق‌تر است و هم کاربردی‌تر.

</div>

<div dir="rtl">

### مرحله ۲۸: Dockerize کردن پروژه

#### در این مرحله پروژه API را داخل یک container قرار می‌دهیم تا بدون وابستگی به محیط سیستم، روی هر سرور یا ماشین دیگری قابل اجرا باشد.

---

#### هدف این مرحله

- ساخت فایل Dockerfile
- تعریف محیط اجرای استاندارد پروژه
- اجرای API داخل container
- آماده‌سازی برای تحویل راحت‌تر به تیم فنی

---

#### اهمیت این مرحله

Docker باعث می‌شود پروژه همراه با تمام وابستگی‌هایش در یک محیط ثابت بسته‌بندی شود.  
به این ترتیب، تفاوت بین سیستم توسعه و سرور نهایی کمتر می‌شود و استقرار پروژه ساده‌تر و قابل اعتمادتر خواهد شد.

</div>

<div dir="rtl">

### مرحله ۲۹: ساخت docker-compose و آماده‌سازی بهتر برای تیم فنی

#### در این مرحله اجرای پروژه را با استفاده از docker-compose ساده‌تر می‌کنیم تا تیم فنی بتواند سرویس را با یک دستور بالا بیاورد و مدیریت کند.

---

#### هدف این مرحله

- ساده‌سازی اجرای پروژه برای تیم فنی
- تعریف تنظیمات container در یک فایل واحد
- کاهش نیاز به نوشتن دستی commandهای طولانی Docker
- آماده‌سازی پروژه برای deploy راحت‌تر

---

#### اهمیت این مرحله

Dockerfile فقط نحوه ساخت image را مشخص می‌کند، اما docker-compose نحوه اجرای سرویس را هم تعریف می‌کند.  
این کار باعث می‌شود اجرای پروژه تکرارپذیر، خوانا و مناسب‌تر برای تیم فنی باشد.

</div>